In [0]:
%sql

SELECT *
FROM dtp_data.dtp_schema.gold_managed_roads_fact
LIMIT 10;

OBJECTID,RD_NAME,RD_TYPE,DEC_NAME,DEC_TYPE,LOCAL_NAME,LOCAL_TYPE,CLASSN,RMACLASS,RD_NUM,RD_SECTION,PROFILE,SRNS,RMANUM,LOCALITY,SEGMENT_LENGTH_M,SEGMENT_LENGTH_KM,DQ_MISSING_DEC_TYPE,DQ_MISSING_RD_TYPE,DQ_MISSING_LOCAL_TYPE,DQ_CORE_ATTRIBUTE_ISSUE,DQ_ZERO_LENGTH,DQ_INVALID_GEOMETRY,DQ_ANY_ISSUE,DQ_OVERALL_STATUS
22700,KATAMATITE-NATHALIA,ROAD,KATAMATITE-NATHALIA,ROAD,KATAMATITE-NATHALIA,ROAD,MR,AO,5401,01,1,C361,5401,NUMURKAH,1100.229871056567,1.100229871056567,false,false,false,false,false,false,false,COMPLETE
22701,KATAMATITE-NATHALIA,ROAD,KATAMATITE-NATHALIA,ROAD,KATAMATITE-NATHALIA,ROAD,MR,AO,5401,01,1,C361,5401,NUMURKAH,46.65517339652992,0.04665517339652992,false,false,false,false,false,false,false,COMPLETE
22702,KATAMATITE-NATHALIA,ROAD,KATAMATITE-NATHALIA,ROAD,KATAMATITE-NATHALIA,ROAD,MR,AO,5401,01,1,C361,5401,NUMURKAH,21.94583033704754,0.02194583033704754,false,false,false,false,false,false,false,COMPLETE
22703,MURRAY VALLEY,HIGHWAY,MURRAY VALLEY,HIGHWAY,MURRAY VALLEY,HIGHWAY,HW,AH,2570,02,1,B400,6570,ESMOND,227.23818905625555,0.22723818905625556,false,false,false,false,false,false,false,COMPLETE
22704,KATAMATITE-NATHALIA,ROAD,KATAMATITE-NATHALIA,ROAD,KATAMATITE-NATHALIA,ROAD,MR,AO,5401,01,1,C361,5401,NUMURKAH,268.2641126854211,0.2682641126854211,false,false,false,false,false,false,false,COMPLETE
22705,KATAMATITE-NATHALIA,ROAD,KATAMATITE-NATHALIA,ROAD,KATAMATITE-NATHALIA,ROAD,MR,AO,5401,01,1,C361,5401,NUMURKAH,283.53778875062727,0.2835377887506273,false,false,false,false,false,false,false,COMPLETE
22706,KATAMATITE-NATHALIA,ROAD,KATAMATITE-NATHALIA,ROAD,KATAMATITE-NATHALIA,ROAD,MR,AO,5401,01,1,C361,5401,NUMURKAH,50.50437604153107,0.05050437604153107,false,false,false,false,false,false,false,COMPLETE
22707,KATAMATITE-NATHALIA,ROAD,KATAMATITE-NATHALIA,ROAD,KATAMATITE-NATHALIA,ROAD,MR,AO,5401,01,1,C361,5401,NUMURKAH,406.84848629896743,0.4068484862989674,false,false,false,false,false,false,false,COMPLETE
22708,KATAMATITE-NATHALIA,ROAD,KATAMATITE-NATHALIA,ROAD,KATAMATITE-NATHALIA,ROAD,MR,AO,5401,01,1,C361,5401,NUMURKAH,44.084166412618416,0.04408416641261841,false,false,false,false,false,false,false,COMPLETE
22709,KATAMATITE-NATHALIA,ROAD,KATAMATITE-NATHALIA,ROAD,KATAMATITE-NATHALIA,ROAD,MR,AO,5401,01,1,C361,5401,NUMURKAH,337.6807295489986,0.33768072954899864,false,false,false,false,false,false,false,COMPLETE


## 1. Gold Fact Table Validation

Confirm the number of records and represented network length in the Gold fact table before performing further analysis.


In [0]:
%sql

SELECT
    COUNT(*) AS road_segments,
    ROUND(SUM(SEGMENT_LENGTH_KM), 2) AS represented_network_km,
    COUNT(DISTINCT RD_NAME) AS road_names,
    COUNT(DISTINCT LOCALITY) AS localities
FROM dtp_data.dtp_schema.gold_managed_roads_fact;

road_segments,represented_network_km,road_names,localities
90797,26297.46,886,2219


## 2. Network by Road Classification

Analyse how the represented managed-road network is distributed across road classifications.

In [0]:
%sql

SELECT
    CLASSN,
    COUNT(*) AS segment_count,
    ROUND(SUM(SEGMENT_LENGTH_KM), 2) AS network_length_km,
    ROUND(
        SUM(SEGMENT_LENGTH_KM) * 100.0 /
        SUM(SUM(SEGMENT_LENGTH_KM)) OVER (),
        2
    ) AS network_length_pct
FROM dtp_data.dtp_schema.gold_managed_roads_fact
GROUP BY CLASSN
ORDER BY network_length_km DESC;

CLASSN,segment_count,network_length_km,network_length_pct
MR,52584,13949.85,53.05
HW,25696,7638.84,29.05
FW,7405,2625.26,9.98
TR,4350,1683.62,6.4
FR,572,356.13,1.35
NR,177,31.45,0.12
PR,13,12.32,0.05


## 3. Network by Management Class

Summarise represented network length and segment counts by road management class.

In [0]:
%sql

SELECT
    RMACLASS,
    COUNT(*) AS segment_count,
    ROUND(SUM(SEGMENT_LENGTH_KM), 2) AS network_length_km
FROM dtp_data.dtp_schema.gold_managed_roads_fact
GROUP BY RMACLASS
ORDER BY network_length_km DESC;

RMACLASS,segment_count,network_length_km
AO,56178,15670.85
AH,27032,7960.76
FW,7397,2622.09
NR,177,31.45
PR,13,12.32


## 4. Top Localities by Represented Network Length

Identify the localities containing the greatest represented managed-road network length.

In [0]:
%sql

SELECT
    LOCALITY,
    COUNT(*) AS segment_count,
    COUNT(DISTINCT RD_NAME) AS road_count,
    ROUND(SUM(SEGMENT_LENGTH_KM), 2) AS network_length_km
FROM dtp_data.dtp_schema.gold_managed_roads_fact
GROUP BY LOCALITY
ORDER BY network_length_km DESC
LIMIT 10;

LOCALITY,segment_count,road_count,network_length_km
OUYEN,137,4,103.9
NHILL,196,5,100.13
BENALLA,312,6,94.15
HOPETOUN,122,4,82.27
NARIEL VALLEY,87,1,80.53
MITTA MITTA,52,2,66.61
WERRIBEE,385,8,65.11
WARRACKNABEAL,130,5,63.87
SUNBURY,346,6,63.84
RAINBOW,74,4,58.9


## 5. Top Roads by Represented Network Length

Identify roads with the greatest represented network length in the dataset.

In [0]:
%sql

SELECT
    RD_NAME,
    COUNT(*) AS segment_count,
    ROUND(SUM(SEGMENT_LENGTH_KM), 2) AS network_length_km
FROM dtp_data.dtp_schema.gold_managed_roads_fact
GROUP BY RD_NAME
ORDER BY network_length_km DESC
LIMIT 10;

RD_NAME,segment_count,network_length_km
PRINCES,4992,1503.46
CALDER,1659,752.67
HUME,1163,673.25
MURRAY VALLEY,1776,647.16
WESTERN,1519,645.87
MIDLAND,1754,448.27
HENTY,748,344.03
SUNRAYSIA,741,338.92
GOULBURN VALLEY,1079,335.58
SOUTH GIPPSLAND,1221,329.55


## 6. Data Quality Status

Assess the proportion of Gold records classified as complete or requiring review.

In [0]:
%sql

SELECT
    DQ_OVERALL_STATUS,
    COUNT(*) AS record_count,
    ROUND(
        COUNT(*) * 100.0 /
        SUM(COUNT(*)) OVER (),
        2
    ) AS record_pct
FROM dtp_data.dtp_schema.gold_managed_roads_fact
GROUP BY DQ_OVERALL_STATUS
ORDER BY record_count DESC;

DQ_OVERALL_STATUS,record_count,record_pct
COMPLETE,89034,98.06
REVIEW,1763,1.94


## 7. Core Attribute Quality Exceptions

Quantify missing core road-type attributes that contribute to records being flagged for review.

In [0]:
%sql

SELECT
    SUM(CASE WHEN DQ_MISSING_RD_TYPE = TRUE THEN 1 ELSE 0 END)
        AS missing_rd_type,

    SUM(CASE WHEN DQ_MISSING_DEC_TYPE = TRUE THEN 1 ELSE 0 END)
        AS missing_dec_type,

    SUM(CASE WHEN DQ_MISSING_LOCAL_TYPE = TRUE THEN 1 ELSE 0 END)
        AS missing_local_type,

    SUM(CASE WHEN DQ_CORE_ATTRIBUTE_ISSUE = TRUE THEN 1 ELSE 0 END)
        AS records_with_core_attribute_issue

FROM dtp_data.dtp_schema.gold_managed_roads_fact;

missing_rd_type,missing_dec_type,missing_local_type,records_with_core_attribute_issue
1321,1321,991,1763


## 8. Data Quality Review by Locality

Identify localities with the largest number of records requiring data-quality review.

In [0]:
%sql

SELECT
    LOCALITY,
    COUNT(*) AS review_records,
    ROUND(SUM(SEGMENT_LENGTH_KM), 2) AS review_network_km
FROM dtp_data.dtp_schema.gold_managed_roads_fact
WHERE DQ_OVERALL_STATUS = 'REVIEW'
GROUP BY LOCALITY
ORDER BY review_records DESC
LIMIT 10;

LOCALITY,review_records,review_network_km
WEST MELBOURNE,225,22.22
MELBOURNE,89,6.62
RINGWOOD,80,15.54
SOUTHBANK,52,3.42
MOUNT MARTHA,51,9.17
WANTIRNA,50,8.3
KEYSBOROUGH,48,13.31
STRATHMORE,44,7.39
DANDENONG NORTH,42,10.48
PASCOE VALE SOUTH,42,5.04


## 9. Records Requiring Review

Provide record-level detail for road segments flagged with core attribute quality issues so that they can be investigated further.

In [0]:
%sql

SELECT
    OBJECTID,
    RD_NAME,
    RD_TYPE,
    DEC_TYPE,
    LOCAL_TYPE,
    CLASSN,
    RMACLASS,
    LOCALITY,
    ROUND(SEGMENT_LENGTH_KM, 3) AS segment_length_km,
    DQ_OVERALL_STATUS
FROM dtp_data.dtp_schema.gold_managed_roads_fact
WHERE DQ_CORE_ATTRIBUTE_ISSUE = TRUE
ORDER BY SEGMENT_LENGTH_KM DESC;

OBJECTID,RD_NAME,RD_TYPE,DEC_TYPE,LOCAL_TYPE,CLASSN,RMACLASS,LOCALITY,segment_length_km,DQ_OVERALL_STATUS
70719,EASTLINK,null,null,null,FW,FW,BANGHOLME,2.737,REVIEW
70721,EASTLINK,null,null,null,FW,FW,BANGHOLME,2.737,REVIEW
71401,EASTLINK,null,null,null,FW,FW,CARRUM DOWNS,2.671,REVIEW
71383,EASTLINK,null,null,null,FW,FW,CARRUM DOWNS,2.359,REVIEW
70151,EASTLINK,null,null,null,FW,FW,DANDENONG SOUTH,1.699,REVIEW
70148,EASTLINK,null,null,null,FW,FW,DANDENONG SOUTH,1.617,REVIEW
48127,PORTARLINGTON-ST LEONARDS,ROAD,ROAD,null,TR,AO,ST LEONARDS,1.255,REVIEW
36473,SUNRAYSIA,HIGHWAY,HIGHWAY,null,HW,AH,DONALD,1.185,REVIEW
41588,MACEDON-WOODEND,ROAD,ROAD,null,MR,AO,WOODEND,1.182,REVIEW
64009,EASTLINK,null,null,null,FW,FW,SCORESBY,1.157,REVIEW
